# ProteinForge — ensayo de rediseño / sequence redesign experiment

**Objetivo:** estructura experimental conocida → secuencias ProteinMPNN → predicción ESMFold de la primera muestra → comparación y ZIP verificable.

**Goal:** known experimental backbone → ProteinMPNN sequences → ESMFold prediction of the first sample → comparison and verifiable ZIP.

Este caso usa ubiquitina pública **1UBQ, cadena A, 76 residuos**. Rediseña una secuencia; **no genera una forma nueva ni demuestra función o estabilidad experimental**. El ensayo completo aún debe ejecutarse y evaluarse. / This public 76-residue case redesigns a sequence, does not generate a new backbone, and does not establish biological function or experimental stability. The full experiment still needs execution and evaluation.

En **Entorno de ejecución → Cambiar tipo de entorno → GPU**, selecciona una GPU gratuita, si está disponible. Después usa **Ejecutar todas**. No contrates recursos. Colab no garantiza GPU ni duración. / Select a free GPU runtime, if available, then Run all. Do not purchase resources; availability and session duration are not guaranteed.

El cálculo se realiza en Colab. No se monta Drive, no se usa Supabase, no se solicita una clave y no se modifica ProteinForge publicado. Los pesos de predicción requieren descargar unos **8,44 GB** en la máquina de Colab. / Computation runs in Colab. No Drive mount, Supabase access, API key or production change is used. Prediction weights download approximately 8.44 GB into the Colab machine.

Versiones y hashes de entradas fijados; Torch/CUDA/GPU del entorno se registran. Repetición de generación con la misma semilla comprobada; igualdad bit a bit entre equipos distintos no garantizada. / Input versions and hashes are pinned; runtime Torch/CUDA/GPU are recorded. Generation is repeated with the same seed; bitwise equality across different devices is not guaranteed.

Fuentes / Sources: [ProteinMPNN](https://github.com/dauparas/ProteinMPNN), [ESMFold](https://huggingface.co/facebook/esmfold_v1), [Transformers ESM](https://huggingface.co/docs/transformers/en/model_doc/esm), [1UBQ](https://www.rcsb.org/structure/1UBQ), [Colab FAQ](https://research.google.com/colaboratory/faq.html).

In [ ]:
#@title Preparar el entorno de Colab / Prepare the Colab environment
PROGRAM_TEXT = '"""Interactive Colab experiment. No web API, external inference or private input."""\n\nfrom __future__ import annotations\n\nimport argparse\nfrom collections import OrderedDict\nfrom contextlib import contextmanager\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nimport traceback\nimport urllib.request\nimport zipfile\n\n\nRESIDUES = dict(zip(\n    "ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split(),\n    "ARNDCQEGHILKMFPSTWYV",\n))\nBACKBONE = {"N", "CA", "C", "O"}\n\n\ndef read_json(path: Path):\n    return json.loads(path.read_text(encoding="utf-8"))\n\n\ndef write_json(path: Path, value):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    text = json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n"\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(text, encoding="utf-8")\n    temporary.replace(path)\n\n\ndef sha256(path: Path):\n    digest = hashlib.sha256()\n    with path.open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef download(url: str, path: Path, expected: str, config: dict):\n    if path.exists() and sha256(path) == expected:\n        return\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".part")\n    request = urllib.request.Request(url, headers={"User-Agent": "ProteinForge-experiment"})\n    with urllib.request.urlopen(request, timeout=config["timeoutSeconds"]) as response:\n        with temporary.open("wb") as stream:\n            shutil.copyfileobj(response, stream, length=config["chunkBytes"])\n    if sha256(temporary) != expected:\n        raise ValueError(f"Downloaded file hash mismatch: {path.name}")\n    temporary.replace(path)\n\n\ndef parse_pdb(text: str, chain: str, expected_sequence: str | None = None):\n    """Read one standard polymer chain in the first model, with explicit residue keys."""\n    residues = OrderedDict()\n    model_seen = False\n    atom_count = 0\n    for line in text.splitlines():\n        if line.startswith("MODEL "):\n            if model_seen:\n                break\n            model_seen = True\n        if line.startswith("ENDMDL"):\n            break\n        if not line.startswith("ATOM  ") or line[21:22] != chain:\n            continue\n        if len(line) < 60:\n            raise ValueError("Truncated PDB atom")\n        if line[16] not in (" ", "A"):\n            continue\n        residue_name = line[17:20].strip()\n        if residue_name not in RESIDUES:\n            raise ValueError("Non-standard polymer residue")\n        coordinates = [float(line[a:b]) for a, b in ((30, 38), (38, 46), (46, 54))]\n        occupancy = float(line[54:60])\n        if not all(math.isfinite(value) for value in (*coordinates, occupancy)):\n            raise ValueError("Non-finite atom data")\n        if occupancy <= 0:\n            continue\n        key = (int(line[22:26]), line[26])\n        residue = residues.setdefault(key, {"name": residue_name, "atoms": {}, "lines": []})\n        if residue["name"] != residue_name:\n            raise ValueError("Ambiguous residue identity")\n        atom = line[12:16].strip()\n        if atom in residue["atoms"]:\n            raise ValueError("Duplicate atom in residue")\n        residue["atoms"][atom] = coordinates\n        residue["lines"].append(line)\n        atom_count += 1\n    if not residues:\n        raise ValueError("Polymer chain is absent")\n    for residue in residues.values():\n        if not BACKBONE <= residue["atoms"].keys():\n            raise ValueError("Incomplete backbone")\n    keys = list(residues)\n    if keys != sorted(keys):\n        raise ValueError("Unordered residue numbering")\n    sequence = "".join(RESIDUES[residue["name"]] for residue in residues.values())\n    if expected_sequence is not None and sequence != expected_sequence:\n        raise ValueError("PDB sequence does not match the exact expected sequence")\n    return {\n        "chain": chain,\n        "sequence": sequence,\n        "residueKeys": [[number, insertion.strip()] for number, insertion in residues],\n        "ca": [residue["atoms"]["CA"] for residue in residues.values()],\n        "atomCount": atom_count,\n        "pdb": "\\n".join(line for residue in residues.values() for line in residue["lines"]) + "\\nTER\\nEND\\n",\n    }\n\n\ndef parse_fasta(text: str):\n    entries = []\n    for line in text.splitlines():\n        line = line.strip()\n        if not line:\n            continue\n        if line.startswith(">"):\n            entries.append({"header": line[1:], "sequence": ""})\n        elif entries:\n            entries[-1]["sequence"] += line\n        else:\n            raise ValueError("FASTA sequence before header")\n    if not entries or any(not entry["sequence"] for entry in entries):\n        raise ValueError("Empty FASTA entry")\n    for entry in entries:\n        if any(aa not in RESIDUES.values() for aa in entry["sequence"]):\n            raise ValueError("Unsupported candidate alphabet or multiple chains")\n    return entries\n\n\ndef compare_ca(reference, prediction):\n    """Least-squares rigid superposition without reflection; coordinates in angstroms."""\n    import numpy as np\n\n    first = np.asarray(reference, dtype=np.float64)\n    second = np.asarray(prediction, dtype=np.float64)\n    if first.shape != second.shape or first.ndim != 2 or first.shape[1] != 3 or len(first) < 3:\n        raise ValueError("Incompatible CA coordinates")\n    if not np.isfinite(first).all() or not np.isfinite(second).all():\n        raise ValueError("Non-finite CA coordinates")\n    first_center = first.mean(axis=0)\n    second_center = second.mean(axis=0)\n    x, y = first - first_center, second - second_center\n    u, _, vt = np.linalg.svd(y.T @ x)\n    correction = np.eye(3)\n    correction[-1, -1] = 1 if np.linalg.det(u @ vt) >= 0 else -1\n    rotation = u @ correction @ vt\n    aligned = y @ rotation + first_center\n    distances = np.linalg.norm(aligned - first, axis=1)\n    return {\n        "method": "Kabsch CA least-squares, proper rotation, no outlier exclusion",\n        "residuesCompared": len(first),\n        "caRmsdAngstrom": float(np.sqrt(np.mean(distances ** 2))),\n        "caDistancesAngstrom": distances.tolist(),\n        "rotationRowVectors": rotation.tolist(),\n        "translation": (first_center - second_center @ rotation).tolist(),\n    }\n\n\ndef gpu_memory_mib():\n    result = subprocess.run(\n        ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],\n        capture_output=True, text=True, timeout=15, check=True,\n    )\n    return max(float(line.strip()) for line in result.stdout.splitlines() if line.strip())\n\n\n@contextmanager\ndef measurement(root: Path, name: str, interval: float):\n    import psutil\n\n    stop = threading.Event()\n    process = psutil.Process()\n    metrics = {"startedAt": datetime.now(timezone.utc).isoformat(), "sampleSeconds": interval,\n               "peakProcessTreeRssBytes": 0, "peakGpuDeviceUsedMiB": None,\n               "samplerErrors": [], "status": "running"}\n\n    def sample():\n        while not stop.is_set():\n            try:\n                rss = sum(p.memory_info().rss for p in [process, *process.children(recursive=True)] if p.is_running())\n                metrics["peakProcessTreeRssBytes"] = max(metrics["peakProcessTreeRssBytes"], rss)\n                used = gpu_memory_mib()\n                metrics["peakGpuDeviceUsedMiB"] = max(metrics["peakGpuDeviceUsedMiB"] or 0, used)\n            except (OSError, subprocess.SubprocessError, psutil.Error, ValueError) as error:\n                if not metrics["samplerErrors"]:\n                    metrics["samplerErrors"].append(type(error).__name__)\n            stop.wait(interval)\n\n    started = time.perf_counter()\n    thread = threading.Thread(target=sample, daemon=True)\n    thread.start()\n    try:\n        yield metrics\n        metrics["status"] = "completed"\n    except BaseException as error:\n        metrics.update(status="failed", errorType=type(error).__name__, error=str(error))\n        raise\n    finally:\n        stop.set()\n        thread.join(timeout=20)\n        metrics["seconds"] = time.perf_counter() - started\n        metrics["finishedAt"] = datetime.now(timezone.utc).isoformat()\n        write_json(root / "measurements" / f"{name}.json", metrics)\n\n\ndef environment(root: Path, config: dict):\n    import torch\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("A managed Colab GPU runtime is required; switch the hardware accelerator to GPU")\n    gpu = torch.cuda.get_device_properties(0)\n    if gpu.total_memory < config["prediction"]["minimumGpuGiB"] * 1024 ** 3:\n        raise RuntimeError("The allocated GPU has insufficient VRAM for this experiment")\n    if shutil.disk_usage(root).free < config["prediction"]["minimumFreeDiskGiB"] * 1024 ** 3:\n        raise RuntimeError("Insufficient free runtime disk space")\n    versions = {}\n    for package in ("torch", "numpy", "transformers", "accelerate", "huggingface-hub", "tokenizers", "psutil", "safetensors"):\n        versions[package] = importlib.metadata.version(package)\n    write_json(root / "environment.json", {\n        "python": sys.version, "platform": platform.platform(), "packages": versions,\n        "torchCuda": torch.version.cuda, "gpu": gpu.name, "gpuMemoryBytes": gpu.total_memory,\n        "freeDiskBytes": shutil.disk_usage(root).free,\n        "experimentScriptSha256": sha256(Path(__file__)),\n        "configSha256": sha256(root / "config.json"),\n    })\n    (root / "pip-freeze.txt").write_text(subprocess.check_output(\n        [sys.executable, "-m", "pip", "freeze"], text=True), encoding="utf-8")\n\n\ndef prepare(root: Path, config: dict):\n    environment(root, config)\n    source = root / "downloads" / "reference.pdb"\n    reference_config = config["reference"]\n    download(reference_config["url"], source, reference_config["sha256"], config["download"])\n    reference = parse_pdb(source.read_text(), reference_config["chain"], reference_config["sequence"])\n    (root / "reference.pdb").write_text(reference["pdb"], encoding="ascii")\n    (root / "reference.fasta").write_text(f">{reference_config[\'id\']} chain {reference_config[\'chain\']} experimental reference\\n{reference[\'sequence\']}\\n")\n    write_json(root / "reference.json", {key: value for key, value in reference.items() if key != "pdb"})\n    for file, digest in config["design"]["files"].items():\n        download(config["design"]["rawBase"] + file, root / "downloads" / "ProteinMPNN" / file,\n                 digest, config["download"])\n    write_json(root / "provenance.json", {\n        "schemaVersion": config["schemaVersion"], "experiment": "known-backbone-sequence-redesign",\n        "reference": reference_config, "design": config["design"], "prediction": config["prediction"],\n        "limitations": config["limitations"],\n    })\n    print("Public reference and pinned ProteinMPNN files verified", flush=True)\n\n\ndef design(root: Path, config: dict):\n    design_config = config["design"]\n    source = root / "downloads" / "ProteinMPNN"\n    reference = read_json(root / "reference.json")\n    outputs = []\n    for run_name in ("design", "design-repeat"):\n        command = [sys.executable, str(source / "protein_mpnn_run.py"),\n                   "--path_to_model_weights", str(source / "vanilla_model_weights"),\n                   "--model_name", design_config["model"], "--pdb_path", str(root / "reference.pdb"),\n                   "--pdb_path_chains", config["reference"]["chain"],\n                   "--out_folder", str(root / run_name), "--seed", str(design_config["seed"]),\n                   "--num_seq_per_target", str(design_config["samples"]), "--batch_size", "1",\n                   "--sampling_temp", str(design_config["temperature"]),\n                   "--max_length", str(design_config["maxLength"]), "--save_score", "1"]\n        write_json(root / "commands" / f"{run_name}.json", command)\n        with measurement(root, run_name, config["measurement"]["sampleSeconds"]):\n            result = subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)\n            (root / f"{run_name}.log").write_text(result.stdout, encoding="utf-8")\n            print(result.stdout, flush=True)\n            result.check_returncode()\n        entries = parse_fasta((root / run_name / "seqs" / "reference.fa").read_text())\n        if entries[0]["sequence"] != reference["sequence"]:\n            raise ValueError("ProteinMPNN reference sequence mismatch")\n        candidates = entries[1:]\n        if len(candidates) != design_config["samples"]:\n            raise ValueError("Unexpected number of generated samples")\n        if any(len(item["sequence"]) != len(reference["sequence"]) for item in candidates):\n            raise ValueError("Candidate length mismatch")\n        outputs.append(candidates)\n    if [item["sequence"] for item in outputs[0]] != [item["sequence"] for item in outputs[1]]:\n        raise RuntimeError("Repeated ProteinMPNN generation did not reproduce the sequences")\n    index = design_config["selectedSample"] - 1\n    candidate = outputs[0][index]\n    candidate.update(sample=index + 1, selectedBeforePrediction=True,\n                     sequenceSha256=hashlib.sha256(candidate["sequence"].encode("ascii")).hexdigest(),\n                     nativeSequenceRecovery=sum(a == b for a, b in zip(candidate["sequence"], reference["sequence"])) / len(reference["sequence"]))\n    write_json(root / "candidate.json", candidate)\n    (root / "candidate.fasta").write_text(f">ProteinForge sample {index + 1}; generated, not experimentally validated\\n{candidate[\'sequence\']}\\n")\n    write_json(root / "design-summary.json", {"candidates": outputs[0], "repeatSequencesEqual": True})\n    print("Candidate selected by fixed sample index; repeat sequence check passed", flush=True)\n\n\ndef predict(root: Path, config: dict):\n    import numpy as np\n    import torch\n    from huggingface_hub import snapshot_download\n    from transformers import EsmConfig, EsmForProteinFolding\n    from transformers.models.esm.openfold_utils import residue_constants\n\n    prediction = config["prediction"]\n    candidate = read_json(root / "candidate.json")\n    sequence = candidate["sequence"]\n    torch.manual_seed(prediction["seed"])\n    torch.cuda.manual_seed_all(prediction["seed"])\n    torch.backends.cudnn.benchmark = False\n    torch.cuda.reset_peak_memory_stats()\n    with measurement(root, "model-download", config["measurement"]["sampleSeconds"]):\n        model_path = Path(snapshot_download(\n            repo_id=prediction["model"], revision=prediction["revision"],\n            allow_patterns=["config.json", prediction["weights"]], token=False,\n        ))\n        weights = model_path / prediction["weights"]\n        if weights.stat().st_size != prediction["weightsBytes"] or sha256(weights) != prediction["weightsSha256"]:\n            raise ValueError("Prediction weights do not match the pinned artifact")\n    with measurement(root, "model-load", config["measurement"]["sampleSeconds"]):\n        model_config = EsmConfig.from_pretrained(model_path, local_files_only=True)\n        model_config.esmfold_config.fp16_esm = True\n        model = EsmForProteinFolding.from_pretrained(\n            model_path, config=model_config, dtype=torch.float16, device_map={"": "cuda:0"},\n            low_cpu_mem_usage=True, local_files_only=True, weights_only=True,\n        ).eval()\n        # Load directly to the GPU in half precision to bound host RAM; only the language model stays half.\n        for name, child in model.named_children():\n            if name != "esm":\n                child.float()\n        for name, parameter in model.named_parameters(recurse=False):\n            parameter.data = parameter.data.float()\n        model.esm.half()\n        model.trunk.set_chunk_size(prediction["chunkSize"])\n        torch.cuda.synchronize()\n    tokens = torch.tensor([[residue_constants.restype_order[aa] for aa in sequence]], device="cuda")\n    with measurement(root, "prediction", config["measurement"]["sampleSeconds"]) as metrics:\n        torch.cuda.reset_peak_memory_stats()\n        with torch.inference_mode():\n            output = model(tokens, attention_mask=torch.ones_like(tokens), num_recycles=prediction["recycles"])\n        torch.cuda.synchronize()\n        metrics["peakTorchAllocatedBytes"] = torch.cuda.max_memory_allocated()\n        metrics["peakTorchReservedBytes"] = torch.cuda.max_memory_reserved()\n        confidence = output.plddt[0, :, 1].float().cpu().numpy()\n        ptm = float(output.ptm[0].item())\n        if not np.isfinite(confidence).all() or not np.all((0 <= confidence) & (confidence <= 1)) or not math.isfinite(ptm) or not 0 <= ptm <= 1:\n            raise ValueError("Invalid confidence output")\n        # This pinned Transformers port returns pLDDT in [0,1]; explicitly export PDB B factors in [0,100].\n        pdb_output = dict(output)\n        pdb_output["plddt"] = output.plddt * 100\n        pdb = model.output_to_pdb(pdb_output)[0]\n        structure = parse_pdb(pdb, "A", sequence)\n        (root / "prediction.pdb").write_text(pdb, encoding="ascii")\n        write_json(root / "prediction.json", {\n            "model": prediction["model"], "revision": prediction["revision"],\n            "sequence": sequence, "sequenceSha256": candidate["sequenceSha256"],\n            "plddtRawScale": [0, 1], "plddtExportScale": [0, 100],\n            "meanCaPlddt": float(confidence.mean() * 100), "caPlddt": (confidence * 100).tolist(),\n            "ptm": ptm, "precision": "ESM fp16, folding trunk and heads fp32",\n            "chain": structure["chain"], "residueCount": len(sequence), "atomCount": structure["atomCount"],\n        })\n        np.savez_compressed(root / "predicted-aligned-error.npz",\n                            pae=output.predicted_aligned_error[0].float().cpu().numpy())\n    print("Exact candidate sequence and finite predicted backbone verified", flush=True)\n\n\ndef package(root: Path, config: dict):\n    required = ("environment.json", "reference.json", "reference.pdb", "candidate.json", "prediction.json", "prediction.pdb", "design-summary.json")\n    missing = [file for file in required if not (root / file).is_file()]\n    validations = {"missing": missing, "status": "incomplete"}\n    # Invalidate a previous validation before checking potentially changed inputs.\n    write_json(root / "validation.json", validations)\n    if not missing:\n        reference = parse_pdb((root / "reference.pdb").read_text(), config["reference"]["chain"], config["reference"]["sequence"])\n        candidate = read_json(root / "candidate.json")\n        recorded_prediction = read_json(root / "prediction.json")\n        sequence_hash = hashlib.sha256(candidate["sequence"].encode("ascii")).hexdigest()\n        if sequence_hash != candidate["sequenceSha256"] or sequence_hash != recorded_prediction["sequenceSha256"] or recorded_prediction["sequence"] != candidate["sequence"]:\n            raise ValueError("Prediction provenance is not associated with the candidate")\n        if not read_json(root / "design-summary.json")["repeatSequencesEqual"]:\n            raise ValueError("Repeated design has not been verified")\n        prediction = parse_pdb((root / "prediction.pdb").read_text(), "A", candidate["sequence"])\n        comparison = compare_ca(reference["ca"], prediction["ca"])\n        write_json(root / "comparison.json", comparison)\n        validations.update(status="completed", exactCandidateSequence=True,\n                           finiteCoordinates=True, completeBackbone=True, repeatedDesign=read_json(root / "design-summary.json")["repeatSequencesEqual"])\n    write_json(root / "validation.json", validations)\n    write_json(root / "result-manifest.json", {\n        "schemaVersion": config["schemaVersion"], "status": validations["status"],\n        "createdAt": datetime.now(timezone.utc).isoformat(), "limitations": config["limitations"],\n        "files": [{"path": str(path.relative_to(root)).replace(os.sep, "/"), "bytes": path.stat().st_size,\n                   "sha256": sha256(path)} for path in sorted(root.rglob("*"))\n                  if path.is_file() and "downloads" not in path.relative_to(root).parts\n                  and path.name != "result-manifest.json"],\n    })\n    archive = root.parent / "proteinforge-design-lab-results.zip"\n    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:\n        for path in sorted(root.rglob("*")):\n            if path.is_file() and "downloads" not in path.relative_to(root).parts:\n                bundle.write(path, arcname=path.relative_to(root))\n    # Reopen every exported entry and verify exact bytes against the manifest.\n    with zipfile.ZipFile(archive) as bundle:\n        if bundle.testzip() is not None:\n            raise ValueError("ZIP integrity check failed")\n        manifest = json.loads(bundle.read("result-manifest.json"))\n        for entry in manifest["files"]:\n            data = bundle.read(entry["path"])\n            if len(data) != entry["bytes"] or hashlib.sha256(data).hexdigest() != entry["sha256"]:\n                raise ValueError("Reopened result hash mismatch")\n    print(json.dumps({"archive": str(archive), "validation": validations}, indent=2), flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("stage", choices=("prepare", "design", "predict", "package"))\n    parser.add_argument("--root", type=Path, required=True)\n    arguments = parser.parse_args()\n    root = arguments.root.resolve()\n    root.mkdir(parents=True, exist_ok=True)\n    if arguments.stage != "package" and (sys.platform != "linux" or not Path("/content").is_dir()):\n        raise RuntimeError("Run the experiment interactively in the managed Colab runtime")\n    try:\n        config = read_json(root / "config.json")\n        globals()[arguments.stage](root, config)\n    except BaseException as error:\n        write_json(root / "failure.json", {"stage": arguments.stage, "type": type(error).__name__,\n                                           "message": str(error), "traceback": traceback.format_exc()})\n        raise\n\n\nif __name__ == "__main__":\n    main()\n'
CONFIG_TEXT = '{\n  "schemaVersion": 1,\n  "reference": {\n    "id": "1UBQ",\n    "chain": "A",\n    "url": "https://files.rcsb.org/download/1UBQ.pdb",\n    "sha256": "d4a6812d8951cf6594e6a0763f089e35f5a80b62acb3c117b2c5565228a7b161",\n    "sequence": "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"\n  },\n  "design": {\n    "repository": "https://github.com/dauparas/ProteinMPNN",\n    "revision": "8907e6671bfbfc92303b5f79c4b5e6ce47cdef57",\n    "rawBase": "https://raw.githubusercontent.com/dauparas/ProteinMPNN/8907e6671bfbfc92303b5f79c4b5e6ce47cdef57/",\n    "files": {\n      "LICENSE": "82009d25ce585631f452b2b24589bdb29c559ccfefa2f200ef312ed5b501a586",\n      "protein_mpnn_run.py": "61f2c519a7f73fa12da9eb90da97b97ec2f8d5f31d42605639c7600cbd321cbe",\n      "protein_mpnn_utils.py": "74c8f9b7553422a7a0bbd705874844ee103c8926c2c96f154a87e0b824071e1b",\n      "vanilla_model_weights/v_48_020.pt": "c9cb4a671d79604111231f8dbfc7c590e06f1197453b7a6854ac6661a642f5bd"\n    },\n    "model": "v_48_020",\n    "seed": 42,\n    "samples": 3,\n    "temperature": 0.1,\n    "selectedSample": 1,\n    "maxLength": 120\n  },\n  "prediction": {\n    "model": "facebook/esmfold_v1",\n    "revision": "75a3841ee059df2bf4d56688166c8fb459ddd97a",\n    "weights": "pytorch_model.bin",\n    "weightsBytes": 8442062570,\n    "weightsSha256": "2ee07356b125d1e3e57503c204111fd7323347fc4735d41d3caac57c2a78e116",\n    "seed": 42,\n    "chunkSize": 32,\n    "recycles": 3,\n    "minimumGpuGiB": 12,\n    "minimumFreeDiskGiB": 20\n  },\n  "measurement": { "sampleSeconds": 0.5 },\n  "download": { "timeoutSeconds": 120, "chunkBytes": 1048576 },\n  "limitations": [\n    "This experiment redesigns a sequence on an experimental backbone; it does not generate a new backbone.",\n    "MPNN likelihood, pLDDT, pTM and backbone RMSD do not demonstrate biological function or experimental stability.",\n    "The selected sample is fixed before prediction; no candidate is selected by predicted confidence.",\n    "Colab runtime versions and hardware are recorded; bitwise reproducibility across different devices is not promised.",\n    "This bundle is an experiment artifact; ProteinForge does not yet import its structures or complete generation history."\n  ]\n}\n'
REQUIREMENTS_TEXT = '# Torch/CUDA are inherited from the managed Colab runtime and recorded in environment.json.\nnumpy==2.2.6\ntransformers==4.57.1\naccelerate==1.11.0\nhuggingface-hub==0.36.0\ntokenizers==0.22.1\nsafetensors==0.6.2\npsutil==7.1.0\n# These optional Transformers imports must remain compatible with NumPy 2.\nlibrosa==0.11.0\n'
#@title Preparar el entorno de Colab / Prepare the Colab environment
import importlib.util
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from datetime import datetime, timezone

if sys.platform != "linux" or importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Open this notebook in a managed Google Colab GPU runtime")
CONFIG = json.loads(CONFIG_TEXT)
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
                     check=True, capture_output=True, text=True).stdout.strip()
print("Allocated GPU:", gpu)
if float(gpu.splitlines()[0].rsplit(",", 1)[1]) / 1024 < CONFIG["prediction"]["minimumGpuGiB"]:
    raise RuntimeError("This notebook needs at least 12 GiB VRAM; do not switch to paid resources")
WORK = Path("/content/proteinforge-design-lab")
WORK.mkdir(exist_ok=True)
if shutil.disk_usage(WORK).free < CONFIG["prediction"]["minimumFreeDiskGiB"] * 1024 ** 3:
    raise RuntimeError("Not enough free space in the managed runtime")
RUN = WORK / datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S%fZ")
RUN.mkdir()
(RUN / "config.json").write_text(CONFIG_TEXT, encoding="utf-8")
(RUN / "experiment.py").write_text(PROGRAM_TEXT, encoding="utf-8")
(RUN / "requirements-colab.txt").write_text(REQUIREMENTS_TEXT, encoding="utf-8")
ENV = WORK / "venv"
# Managed Debian runtimes may omit ensurepip; inherit their existing pip instead.
subprocess.run([sys.executable, "-m", "venv", "--without-pip", "--system-site-packages", str(ENV)], check=True)
PYTHON = str(ENV / "bin" / "python")
subprocess.run([PYTHON, "-m", "pip", "install", "--disable-pip-version-check",
                "--report", str(RUN / "installation-report.json"),
                "-r", str(RUN / "requirements-colab.txt")], check=True)
STAGE_ENV = dict(os.environ, HF_HOME=str(WORK / "model-cache"),
                 HF_HUB_DISABLE_IMPLICIT_TOKEN="1", HF_HUB_DISABLE_TELEMETRY="1", USE_TF="0", USE_FLAX="0")

def run_stage(name):
    command = [PYTHON, "-u", str(RUN / "experiment.py"), name, "--root", str(RUN)]
    print("Stage:", name, "| Results:", RUN, flush=True)
    result = subprocess.run(command, env=STAGE_ENV)
    if result.returncode:
        raise RuntimeError(f"Stage {name} failed. Read its output; the final download cell can export partial evidence.")
print("Environment ready. No private sequence or Drive mount is used.")

## Referencia e integridad / Reference and integrity
Descarga solo datos públicos y archivos oficiales con revisión y SHA-256 fijados. Valida la cadena y todos los átomos del esqueleto. / Download public data and pinned official files; validate the chain and complete backbone atoms.

In [ ]:
run_stage("prepare")

## Rediseño / Sequence redesign
Genera tres secuencias y repite con la misma semilla. La muestra 1 se selecciona **antes de predecir**, sin elegirla por su confianza posterior. El score de ProteinMPNN es una puntuación del modelo, no energía física. / Generate three sequences, repeat with the same seed, and select sample 1 before prediction. ProteinMPNN scores are model likelihood scores, not physical energy.

In [ ]:
run_stage("design")
print((RUN / "candidate.fasta").read_text())

## Predicción / Prediction
ESMFold recibe la secuencia exacta seleccionada. Valida coordenadas finitas, residuos y esqueleto completo. Registra pLDDT, pTM, memoria y tiempo. Este port entrega pLDDT en 0–1; el PDB y el resumen se exportan explícitamente en 0–100. No es probabilidad de función. / ESMFold receives the exact selected sequence; coordinates, residues and backbone completeness are validated. Confidence, memory and time are recorded; this port's 0–1 pLDDT is explicitly exported on a 0–100 scale. It is not a probability of function.

In [ ]:
run_stage("predict")
print((RUN / "prediction.json").read_text())

## Comparar y descargar / Compare and download
RMSD de Cα tras superposición rígida sin reflexión, sin excluir residuos. No comparar como si ambas secuencias fueran idénticas: cada posición corresponde al esqueleto de referencia usado para diseñar. El ZIP excluye pesos/cachés y se reabre para verificar sus hashes. / Cα RMSD after rigid superposition without reflection or residue exclusion. Positional correspondence comes from the design backbone, not from identical sequences. Weights and caches are excluded; the ZIP is reopened and its hashes checked.

Si una etapa falla, ejecuta esta última celda para descargar evidencia parcial: figurará **incomplete**, sin inventar resultados. ProteinForge todavía no importa automáticamente este paquete. / If a stage fails, run this final cell to export partial evidence marked incomplete. ProteinForge does not yet import this bundle automatically.

In [ ]:
run_stage("package")
if (RUN / "comparison.json").exists():
    print((RUN / "comparison.json").read_text())
print((RUN / "validation.json").read_text())
from google.colab import files
files.download(str(WORK / "proteinforge-design-lab-results.zip"))